In [ ]:
import datetime as dt
import os

import xarray as xr
from pyproj import CRS

In [ ]:
from etdataset.dem import get_dem_from_roi
from etdataset.era5 import ERA5Dataset
from etdataset.icos import get_stations_config
from etdataset.utils import work_area_from_coord_point
from etdataset.validation_temp.temperature_rescaling import (
    create_era5_sub_dataset,
    filter_dataset_by_hours,
    generate_hours,
    get_ta_td_celsius_at_location,
    prepare_temperature_inputs,
    read_era5_file,
    save_ta_td_csv,
    temperature_rescaling_constant_lapse_rate,
)

In [ ]:
from etdataset.logging import LoggerManager

logger = LoggerManager.get_logger(__name__)


# Comparison temperature rescaling

## Prepare inputs

### Generate a list of times for the desired day

In [ ]:
date = dt.date(2023, 3, 3)

In [ ]:
list_hours = generate_hours()

### Read one ERA5 file

In [ ]:
era5_xrds = read_era5_file(date, ERA5Dataset.ERA5, "out")

### Filter dataset to keep only the desired hours

In [ ]:
era5_xrds = filter_dataset_by_hours(era5_xrds, date, list_hours)

In [ ]:
era5_xrds

### Get metadatas of a desired station

In [ ]:
cfg = get_stations_config("DE-Geb")

In [ ]:
cfg

## Extract air temperature (TA) and dew point temperature (TD) at the station location from ERA5 data 

### Extract air temperature (Ta) and dew point temperature (Td) in Celsius at station's location

In [ ]:
era5_sub = create_era5_sub_dataset(era5_xrds)

In [ ]:
era5_sub

In [ ]:
ta, td = get_ta_td_celsius_at_location(era5_sub, cfg)

# Extract TA and TD at the station location from DEM-adjusted ERA5 temperatures.

### Get a ROI around the station

In [ ]:
roi_bbox_utm, roi_crs_utm = work_area_from_coord_point(
    cfg.lat, cfg.lon, 10000, 10000, CRS.from_epsg(4326)
)["utm"]


In [ ]:
mnt_path = os.environ["MNT_PATH"]

### Get a DEM over the ROI

In [ ]:
dem = get_dem_from_roi(
    roi_bbox=roi_bbox_utm,
    roi_crs=roi_crs_utm,
    base_dir=mnt_path,
    resolution=60,
)

In [ ]:
dem

### Prepare datas for the temperature rescaling

In [ ]:
dem, era5_dem, updated_data = prepare_temperature_inputs(
    data=dem,
    era5_data=era5_xrds,
    dataset=ERA5Dataset.ERA5,
)


### Adjust ERA5 temperatures (air and dew point) to the DEM using a constant lapse rate.

In [ ]:
updated = temperature_rescaling_constant_lapse_rate(
    updated_data,
    dem,
    era5_xrds,
    era5_dem,
)


In [ ]:
updated

### Extract air temperature (Ta) and dew point temperature (Td) in Celsius at station's location

In [ ]:
ta, td = get_ta_td_celsius_at_location(updated, cfg)


In [ ]:
ta

In [ ]:
td

In [ ]:
ds_out = xr.Dataset(
    {
        "ta": ta,
        "tdp": td,
    }
)


### Save air temperature (TA) and dew point temperature (TD) data to a CSV file.

In [ ]:
save_ta_td_csv(ds_out, cfg)